# Stage 1: FITS, Units, and Time

This offline notebook demonstrates FITS, units, and time. If `data/raw` has no real files,
it uses a synthetic TESS-format light curve.

Use the **IERAS** Jupyter kernel.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import astropy.units as u
from astropy.time import Time
from astropy.io import fits
import numpy as np

print("Python:", sys.version.split()[0])
print("Project root:", ROOT)

## 1. Units

Store units with values and use Astropy for conversions.

In [ ]:
star_radius = 0.95 * u.R_sun
planet_radius = 1.2 * u.R_earth

print(star_radius.to(u.R_earth))
print(planet_radius.to(u.km))

# Adding incompatible units raises an error, preventing invalid calculations.
try:
    planet_radius + 5 * u.kg
except u.UnitConversionError as exc:
    print("Unit error:", exc)

## 2. Time scales

JD, BJD, MJD, BTJD, and BKJD are not interchangeable. Read the headers first.

In [ ]:
t = Time(59000, format="mjd", scale="tdb")
print("JD:", t.jd)
print("ISO:", t.iso)

# For TESS, TIME is BTJD = BJD_TDB - 2457000. Read the epoch from BJDREFI.
print("Example: BTJD 1500.0 with BJDREFI=2457000 corresponds to JD", 2457000 + 1500.0)

## 3. FITS HDUs and headers

Inspect the file without modifying the source.

In [ ]:
from src.utils.fits_utils import format_report, inspect_lightcurve
from tests.fakes import make_tess_like_lc

raw_files = sorted((ROOT / "data" / "raw").rglob("*.fits"))
if raw_files:
    sample = raw_files[0]
    print("Real file:", sample)
else:
    sample = make_tess_like_lc(ROOT / "reports" / "synthetic_lc.fits")
    print("No real files found; using synthetic file:", sample)

with fits.open(sample, memmap=False) as hdul:
    hdul.info()
    lc = hdul["LIGHTCURVE"] if "LIGHTCURVE" in hdul else hdul[1]
    print("Columns:", lc.columns.names)
    print("TIMESYS:", lc.header.get("TIMESYS"), "| BJDREFI:", lc.header.get("BJDREFI"))

print()
print(format_report(inspect_lightcurve(sample)))

## 4. Next steps

- Read VOTable files with `astropy.table.Table.read(..., format="votable")`.
- Spectrum units and transmission spectra are planned for a later stage.
- The same inspector checks Kepler light curves (BKJD, reference epoch 2454833).